# Activation Steering A/B for Toy Flow Matching

This notebook is a polished continuation of the toy notebooks in this folder. It adds the next set of experiments for steering dynamics inspired by arXiv:2602.11395.

We compare

- baseline flow matching,
- time-windowed steering with constant/triangular/cosine profiles,
- a compact ablation grid over steering strength and schedule.

The setup is intentionally CPU-friendly and keeps all operations on 2D toy targets.


In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torchdyn.core import NeuralODE


In [ ]:
SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
def sample_noise(n):
    return torch.randn(n, 2, device=device)

def safe_float(x):
    return float(torch.as_tensor(x).detach().cpu().item())

def ensure_finite(name, tensor_like):
    arr = torch.as_tensor(tensor_like, dtype=torch.float32)
    if not torch.isfinite(arr).all():
        raise ValueError(f"Non-finite values detected in {name}")

def to_numpy(x):
    return x.detach().cpu().numpy()

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 200,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
})


In [ ]:
# Toy targets

def sample_circle_discrete(n, n_pts=12):
    angles = torch.linspace(0, 2 * math.pi, n_pts + 1, device=device)[:-1]
    xi = torch.stack([torch.cos(angles), torch.sin(angles)], dim=1)
    idx = torch.randint(0, n_pts, (n,), device=device)
    return xi[idx]

def sample_disk_uniform(n):
    theta = 2 * math.pi * torch.rand(n, device=device)
    r = torch.sqrt(torch.rand(n, device=device))
    return torch.stack([r * torch.cos(theta), r * torch.sin(theta)], dim=1)

def sample_swiss_roll(n, noise=0.3):
    t = 1.5 * math.pi * (1 + 2 * torch.rand(n, device=device))
    x = t * torch.cos(t) + noise * torch.randn(n, device=device)
    y = t * torch.sin(t) + noise * torch.randn(n, device=device)
    pts = torch.stack([x, y], dim=1)
    return pts / pts.abs().max() * 2

def sample_ring(n, center=(0.0, 0.0), radius=1.0, width=0.15):
    center = torch.as_tensor(center, device=device)
    theta = 2 * math.pi * torch.rand(n, device=device)
    r = radius + width * (torch.rand(n, device=device) - 0.5)
    return torch.stack([r * torch.cos(theta), r * torch.sin(theta)], dim=1) + center

TARGET_SPECS = {
    "circle": (sample_circle_discrete, torch.tensor([0.0, 0.0], device=device)),
    "disk": (sample_disk_uniform, torch.tensor([0.0, 0.0], device=device)),
    "swiss": (sample_swiss_roll, torch.tensor([0.0, 0.0], device=device)),
    "ring": (lambda n: sample_ring(n, radius=1.0, width=0.08), torch.tensor([0.0, 0.0], device=device)),
}

def get_target(name):
    return TARGET_SPECS[name]


In [ ]:
class VelocityMLP(nn.Module):
    def __init__(self, dim=2, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, hidden), nn.SELU(),
            nn.Linear(hidden, hidden), nn.SELU(),
            nn.Linear(hidden, hidden), nn.SELU(),
            nn.Linear(hidden, dim),
        )

    def forward(self, t, x, args=None):
        if t.dim() == 0:
            t = t.repeat(x.shape[0])
        return self.net(torch.cat([x, t[:, None]], dim=-1))


def ot_constant(x0, x1, t):
    t = t[:, None]
    return t * x1 + (1 - t) * x0, x1 - x0


class ODEWrapper(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, t, x, args=None):
        return self.net(t.repeat(x.shape[0]), x)


def train(sample_target_fn, n_steps=1500, batch_size=256, lr=1e-3, print_every=500):
    model = VelocityMLP().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []

    t0 = time.time()
    for step in range(1, n_steps + 1):
        x0 = sample_noise(batch_size)
        x1 = sample_target_fn(batch_size).to(device)
        t = torch.rand(batch_size, device=device)

        xt, ut = ot_constant(x0, x1, t)
        pred = model(t, xt)
        loss = ((pred - ut) ** 2).mean()

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(safe_float(loss))
        if step % print_every == 0:
            elapsed = time.time() - t0
            t0 = time.time()
            print(f"step {step:>5d} | loss {loss.item():.4f} | {elapsed:.1f}s")

    return model, losses


In [ ]:
def integrate_model(model, n_samples, n_time, wrap_fn=None, wrap_kwargs=None):
    if wrap_fn is None:
        ode = NeuralODE(ODEWrapper(model), solver="midpoint", sensitivity="adjoint")
    else:
        ode = NeuralODE(wrap_fn(model=model, **(wrap_kwargs or {})), solver="midpoint", sensitivity="adjoint")

    with torch.no_grad():
        traj = ode.trajectory(
            sample_noise(n_samples),
            t_span=torch.linspace(0, 1, n_time, device=device),
        )
    return traj


def plot_loss(losses, title):
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(losses, alpha=0.3, lw=0.8)
    w = 200
    if len(losses) > w:
        ma = np.convolve(losses, np.ones(w) / w, mode="valid")
        ax.plot(range(w - 1, len(losses)), ma, lw=2, label=f"moving avg ({w})")
    ax.set_title(f"{title} | train loss")
    ax.set_xlabel("step")
    ax.set_ylabel("MSE")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.show()


In [ ]:
def steering_gate(t, start=0.2, end=0.8, profile="triangular"):
    t = t[:, None]
    active = ((t >= start) & (t <= end)).float()

    if profile == "constant":
        return active

    span = max(end - start, 1e-8)
    u = ((t - start) / span).clamp(0.0, 1.0)

    if profile == "triangular":
        w = torch.minimum(u, 1 - u).clamp(min=0.0) * 2
        return w * active
    if profile == "cosine":
        left = 0.5 - 0.5 * torch.cos(2 * math.pi * u)
        right = 0.5 - 0.5 * torch.cos(2 * math.pi * (1 - u))
        raw = torch.where(u <= 0.5, left, right)
        return raw * active

    raise ValueError("profile must be constant, triangular, or cosine")


def steered_velocity(model, x, t, anchor, steer_strength, start, end, profile):
    base = model(t, x)
    if steer_strength == 0.0:
        return base

    target = torch.as_tensor(anchor, device=x.device, dtype=x.dtype)
    if target.ndim == 1:
        target = target[None, :].expand_as(x)

    w = steering_gate(t, start=start, end=end, profile=profile)
    return base + steer_strength * w * torch.tanh(target - x)


class SteeredODEWrapper(nn.Module):
    def __init__(self, model, anchor, steer_strength, start, end, profile):
        super().__init__()
        self.model = model
        self.anchor = anchor
        self.steer_strength = float(steer_strength)
        self.start = float(start)
        self.end = float(end)
        self.profile = profile

    def forward(self, t, x, args=None):
        tt = t.repeat(x.shape[0]) if t.dim() == 0 else t
        return steered_velocity(
            self.model,
            x,
            tt,
            anchor=self.anchor,
            steer_strength=self.steer_strength,
            start=self.start,
            end=self.end,
            profile=self.profile,
        )


In [ ]:
def trajectory_length(traj):
    d = torch.diff(traj, dim=0)
    step_len = torch.linalg.norm(d, dim=-1)
    return step_len.sum(dim=0)


def nearest_target_distance(samples, target_anchor):
    target = torch.as_tensor(target_anchor, device=samples.device, dtype=samples.dtype)
    if target.ndim == 1:
        target = target[None, :]
    dmat = torch.cdist(samples, target)
    nearest = dmat.min(dim=1).values
    return nearest.mean(), nearest.quantile(0.9), nearest.max()


def run_trajectory_experiment(traj, target_anchor, name):
    end = traj[-1]
    ensure_finite(name, end)
    mean, p90, maxv = nearest_target_distance(end, target_anchor)
    return {
        "name": name,
        "mean_dist": float(mean),
        "p90_dist": float(p90),
        "max_dist": float(maxv),
        "path_length": float(trajectory_length(traj).mean()),
        "final_mse": float(torch.mean(torch.sum((end - target_anchor) ** 2, dim=-1)).item()),
    }


def run_ablations(model, target_anchor, strengths=(0.0, 0.2, 0.4), windows=((0.15, 0.85), (0.30, 0.70)), profiles=("constant", "triangular", "cosine"), n_samples=1200, n_time=180):
    rows = []
    for strength in strengths:
        for start, end in windows:
            for profile in profiles:
                if strength == 0.0 and profile != "constant":
                    continue
                t0 = time.time()
                if strength == 0.0:
                    traj = integrate_model(model, n_samples=n_samples, n_time=n_time)
                else:
                    traj = integrate_model(
                        model,
                        n_samples=n_samples,
                        n_time=n_time,
                        wrap_fn=SteeredODEWrapper,
                        wrap_kwargs={
                            "anchor": target_anchor,
                            "steer_strength": strength,
                            "start": start,
                            "end": end,
                            "profile": profile,
                        },
                    )

                result = run_trajectory_experiment(traj, target_anchor, f"s{strength}_{start}_{end}_{profile}")
                result.update({
                    "strength": float(strength),
                    "start": float(start),
                    "end": float(end),
                    "profile": profile,
                    "wall_ms": (time.time() - t0) * 1000.0,
                    "traj": to_numpy(traj),
                })
                rows.append(result)
    return rows


In [ ]:
# Baseline over four targets
BASELINE_STEPS = 1200
BASELINE_BATCH = 256

experiment_state = {}

for name in ["circle", "disk", "swiss", "ring"]:
    sampler, anchor = get_target(name)
    print(f"\n==== Baseline on {name} ====")
    t_start = time.time()
    model, losses = train(sampler, n_steps=BASELINE_STEPS, batch_size=BASELINE_BATCH, print_every=400)
    traj = integrate_model(model, n_samples=2048, n_time=220)
    metrics = run_trajectory_experiment(traj, anchor, f"{name} baseline")
    metrics["target"] = name
    metrics["train_s"] = time.time() - t_start
    metrics["loss_last"] = losses[-1]

    experiment_state[name] = {
        "sampler": sampler,
        "anchor": anchor,
        "model": model,
        "traj": to_numpy(traj),
        "losses": losses,
        "metrics": metrics,
    }

    plot_loss(losses, name)
    print(metrics)


## Baseline target snapshots


In [ ]:
def plot_baseline_endpoints(experiment_state, n_show=900):
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    axes = axes.ravel()

    for ax, (name, st) in zip(axes, experiment_state.items()):
        xs = st["traj"][-1, :n_show]
        ax.scatter(xs[:, 0], xs[:, 1], s=5, alpha=0.45, label="generated")
        ax.scatter(st["anchor"][0].item(), st["anchor"][1].item(), s=80, c="red", marker="*", edgecolors="k", label="anchor")
        ax.set_title(name)
        ax.set_aspect('equal')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.legend(loc='best')
    plt.tight_layout()
    plt.show()

plot_baseline_endpoints(experiment_state)


## Steering ablations


In [ ]:
rows = []

for target in ["circle", "disk", "swiss", "ring"]:
    model = experiment_state[target]["model"]
    _, anchor = get_target(target)

    res = run_ablations(
        model,
        target_anchor=anchor,
        strengths=(0.0, 0.2, 0.5),
        windows=((0.2, 0.8), (0.3, 0.7)),
        profiles=("constant", "triangular", "cosine"),
        n_samples=1200,
        n_time=180,
    )

    for r in res:
        r["target"] = target
        rows.append(r)

print(f"Collected {len(rows)} steering rows")


In [ ]:
def plot_trajectory_grid(rows, n_lines=20):
    # 4x2 layout: each target compares baseline and one non-zero steering setting
    by_target = {}
    for r in rows:
        by_target.setdefault(r["target"], []).append(r)

    fig, axes = plt.subplots(4, 2, figsize=(10, 16))
    for row, target in enumerate(["circle", "disk", "swiss", "ring"]):
        entries = by_target.get(target, [])
        base = next((r for r in entries if r["strength"] == 0.0), None)
        steer = next((r for r in entries if r["strength"] >= 0.5), None)

        for col, r in enumerate([base, steer]):
            ax = axes[row, col]
            if r is None:
                ax.set_axis_off()
                continue

            traj = np.asarray(r["traj"])
            idx = np.random.choice(traj.shape[1], size=min(n_lines, traj.shape[1]), replace=False)
            for j in idx:
                ax.plot(traj[:, j, 0], traj[:, j, 1], lw=0.7, alpha=0.45)
            ax.scatter(traj[0, :, 0], traj[0, :, 1], s=8, c="gray", alpha=0.25)
            ax.scatter(traj[-1, :, 0], traj[-1, :, 1], s=8, c="tab:blue", alpha=0.35)
            ax.set_aspect('equal')
            ax.set_xticks([])
            ax.set_yticks([])
            label = "baseline" if r["strength"] == 0.0 else f"s={r['strength']} {r['profile']}"
            ax.set_title(f"{target} | {label}")
    plt.tight_layout()
    plt.show()

plot_trajectory_grid(rows, n_lines=18)


In [ ]:
print("Top configs by mean anchor distance per target")

for target in ["circle", "disk", "swiss", "ring"]:
    sub = [r for r in rows if r["target"] == target]
    sub = sorted(sub, key=lambda x: (x["mean_dist"], x["path_length"]))
    print(f"\n{target}")
    for r in sub[:4]:
        print(
            f"  strength={r['strength']:.2f}, window={r['start']:.2f}-{r['end']:.2f}, profile={r['profile']:<9} | "            f"mean={r['mean_dist']:.4f}, p90={r['p90_dist']:.4f}, length={r['path_length']:.4f}, mse={r['final_mse']:.4f}, ms={r['wall_ms']:.0f}"
        )


## Interpretation notes

- Start with a tiny steering run: `strength=0.2, window=(0.3, 0.7), profile='triangular'`
- Keep baseline loss quality stable first, then compare trajectory quality metrics
- If mean and p90 distances improve but final path length explodes, the steering is too aggressive

Next iterations can replace the anchor with a learned semantic direction and connect these controls to actual activation steering in higher-dimensional latent generators.
